# Self-Supervised Learning and Imaging Foundation Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parisar/BME6938-Advanced-Biomedical-AI-Spring-2027/blob/main/lectures/notebooks/week06_self_supervised_learning.ipynb)

*Click the badge to open this notebook in Google Colab, then choose File > Save a copy in Drive so your changes are kept.*

**BME 6938 Advanced Biomedical AI · Week 6 · Reference notebook (ungraded)**

This notebook follows Lecture 6-1 on real pathology images: **PathMNIST**, small (28 × 28) tiles from colorectal cancer
histology slides in 9 tissue classes. We pretrain two tiny encoders **without labels**, then measure what they learned with
a **linear probe** using only a few labels, exactly as in the lecture.

| Section | Lecture step | What you will see |
|---|---|---|
| 1 | The label bottleneck | The dataset, and how few labels we will allow ourselves |
| 2 | Masked reconstruction, steps 1 to 3 | Tiles cut into patches; 75% of patches hidden |
| 3 | Masked reconstruction, steps 4 and 5 | A tiny masked autoencoder, trained without labels |
| 4 | Contrastive learning | Two augmented views, the InfoNCE loss with the slide numbers, a tiny SimCLR |
| 5 | Using the encoder | Linear probes with 1%, 10%, and 100% of the labels |
| 6 | Imaging foundation models (optional) | Features from a pretrained DINOv2 encoder |

**How to run it**
- **Google Colab:** *Runtime > Run all*. CPU works (about 10 minutes); a GPU makes it faster.
- **HiPerGator:** a PyTorch kernel in Jupyter. Set `DATA_DIR` in the setup cell to a folder on your group's `/blue` storage.
- The first run downloads PathMNIST (about 200 MB) and, in the optional section 6, DINOv2-small (about 90 MB).
  If the download fails, the notebook falls back to synthetic tiles so every cell still runs (results will differ).

Models here are deliberately tiny so they train on a CPU in minutes. Real foundation models use the same code ideas at a
scale of millions of slides and thousands of GPU hours.

## 0. Setup

In [ ]:
import importlib, subprocess, sys, os, math, time
for package, pip_name in [("torch", "torch"), ("sklearn", "scikit-learn"), ("medmnist", "medmnist")]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=False)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

DATA_DIR = os.path.expanduser("~/medmnist_data")   # on HiPerGator, use a folder on /blue
os.makedirs(DATA_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)
print("PyTorch", torch.__version__, "| device:", DEVICE)

## 1. The label bottleneck  *(slides: "The Label Bottleneck")*

We load PathMNIST and pretend that labels are expensive: pretraining uses the **images only**, and the labels are used later,
in small amounts, for the linear probe. To keep CPU training fast we use a subset of the training set.

In [ ]:
def load_pathmnist(n_train=8000, n_test=2000):
    # Return (train_images, train_labels, test_images, test_labels, class_names); images are float tensors (N, 3, 28, 28) in [0, 1].
    try:
        from medmnist import PathMNIST
        train = PathMNIST(split="train", download=True, root=DATA_DIR)
        test = PathMNIST(split="test", download=True, root=DATA_DIR)
        to_tensor = lambda a: torch.tensor(a, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
        names = [train.info["label"][str(i)] for i in range(9)]
        return (to_tensor(train.imgs[:n_train]), torch.tensor(train.labels[:n_train, 0]),
                to_tensor(test.imgs[:n_test]), torch.tensor(test.labels[:n_test, 0]), names)
    except Exception as error:
        print("PathMNIST download failed (", type(error).__name__, "): using synthetic tiles instead.")
        return synthetic_tiles(n_train, n_test)

def synthetic_tiles(n_train, n_test, n_classes=3):
    # Fallback data: pink 'tissue' tiles whose classes differ in nucleus density and color. Only for running offline.
    g = torch.Generator().manual_seed(0)
    def make(n):
        y = torch.randint(0, n_classes, (n,), generator=g)
        x = torch.ones(n, 3, 28, 28) * torch.tensor([0.95, 0.75, 0.88]).view(1, 3, 1, 1)
        yy, xx = torch.meshgrid(torch.arange(28), torch.arange(28), indexing="ij")
        for i in range(n):
            for _ in range(4 + 6 * int(y[i])):
                cy, cx = torch.randint(2, 26, (2,), generator=g)
                blob = ((yy - cy) ** 2 + (xx - cx) ** 2) < 4
                x[i, :, blob] = torch.tensor([0.45, 0.2 + 0.1 * int(y[i]), 0.6]).view(3, 1)
        return (x + 0.03 * torch.randn(x.shape, generator=g)).clamp(0, 1), y
    xtr, ytr = make(n_train); xte, yte = make(n_test)
    return xtr, ytr, xte, yte, [f"synthetic class {i}" for i in range(n_classes)]

X_train, y_train, X_test, y_test, CLASS_NAMES = load_pathmnist()
N_CLASSES = len(CLASS_NAMES)
print("train", tuple(X_train.shape), "| test", tuple(X_test.shape), "| classes:", N_CLASSES)

fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
for ax, i in zip(axes, range(8)):
    ax.imshow(X_train[i].permute(1, 2, 0)); ax.set_title(CLASS_NAMES[int(y_train[i])][:14], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Masked reconstruction, steps 1 to 3  *(slides: "Step 1" to "Step 3")*

**Step 1: cut into patches.** A 28 × 28 tile with 4 × 4 patches gives 7 × 7 = 49 patches. Each patch (4 × 4 × 3 = 48 numbers)
is one token, as in a ViT.

**Step 2: hide most of them.** We hide 75% of the patches at random, as MAE does.

**Step 3: encode only the visible ones.** The encoder below will receive only the 12 visible patches.

In [ ]:
PATCH = 4
N_PATCHES = (28 // PATCH) ** 2            # 49
PATCH_DIM = PATCH * PATCH * 3             # 48 numbers per patch

def patchify(images):
    # (B, 3, 28, 28) -> (B, 49, 48): one row per patch, left to right, top to bottom.
    B = images.shape[0]
    p = images.unfold(2, PATCH, PATCH).unfold(3, PATCH, PATCH)          # (B, 3, 7, 7, 4, 4)
    return p.permute(0, 2, 3, 1, 4, 5).reshape(B, N_PATCHES, PATCH_DIM)

def unpatchify(patches):
    # (B, 49, 48) -> (B, 3, 28, 28), the inverse of patchify.
    B, g = patches.shape[0], 28 // PATCH
    p = patches.reshape(B, g, g, 3, PATCH, PATCH).permute(0, 3, 1, 4, 2, 5)
    return p.reshape(B, 3, 28, 28)

def random_mask(batch_size, mask_ratio=0.75):
    # For each image, return indices of visible and hidden patches (a new random choice every call).
    n_keep = int(N_PATCHES * (1 - mask_ratio))
    order = torch.rand(batch_size, N_PATCHES).argsort(dim=1)
    return order[:, :n_keep], order[:, n_keep:]                         # visible, hidden

patches = patchify(X_train[:1])
visible, hidden = random_mask(1)
print("patches per tile:", N_PATCHES, "| visible:", visible.shape[1], "| hidden:", hidden.shape[1])

masked = patches.clone(); masked[0, hidden[0]] = 0.8                     # grey out hidden patches for display
fig, axes = plt.subplots(1, 2, figsize=(5, 2.6))
axes[0].imshow(X_train[0].permute(1, 2, 0)); axes[0].set_title("tile")
axes[1].imshow(unpatchify(masked)[0].permute(1, 2, 0)); axes[1].set_title("75% of patches hidden")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Masked reconstruction, steps 4 and 5  *(slides: "Step 4" and "Step 5")*

A tiny masked autoencoder:
- the **encoder** (a small transformer) sees only the visible patches plus their position vectors;
- the **decoder** gets the encoded visible patches plus a learned **mask token** at every hidden position, and predicts the pixels;
- the **loss** is the mean squared error on the **hidden patches only**.

After training we keep the encoder and discard the decoder.

In [ ]:
class TinyMAE(nn.Module):
    def __init__(self, d=64, depth=2, heads=4):
        super().__init__()
        self.embed = nn.Linear(PATCH_DIM, d)                                # patch -> token
        self.pos = nn.Parameter(torch.randn(1, N_PATCHES, d) * 0.02)        # learned position vectors
        layer = lambda: nn.TransformerEncoderLayer(d, heads, 4 * d, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer(), depth)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d))                # stands in for every hidden patch
        self.decoder = nn.TransformerEncoder(layer(), 1)                    # deliberately small
        self.to_pixels = nn.Linear(d, PATCH_DIM)

    def encode(self, images, visible=None):
        # Encode the visible patches (all patches if visible is None). Returns (B, n_visible, d).
        tokens = self.embed(patchify(images)) + self.pos
        if visible is not None:
            tokens = torch.gather(tokens, 1, visible[..., None].expand(-1, -1, tokens.shape[-1]))
        return self.encoder(tokens)

    def forward(self, images, mask_ratio=0.75):
        B = images.shape[0]
        visible, hidden = random_mask(B, mask_ratio)
        visible, hidden = visible.to(images.device), hidden.to(images.device)
        encoded = self.encode(images, visible)                              # step 3: visible patches only
        full = self.mask_token.expand(B, N_PATCHES, -1).clone()             # step 4: mask tokens everywhere ...
        full.scatter_(1, visible[..., None].expand(-1, -1, full.shape[-1]), encoded)   # ... then put back the visible ones
        predicted = self.to_pixels(self.decoder(full + self.pos))           # (B, 49, 48) predicted pixels
        target = patchify(images)
        pick = lambda t: torch.gather(t, 1, hidden[..., None].expand(-1, -1, PATCH_DIM))
        loss = F.mse_loss(pick(predicted), pick(target))                    # hidden patches only
        return loss, predicted, hidden

mae = TinyMAE().to(DEVICE)
optimizer = torch.optim.AdamW(mae.parameters(), lr=2e-3, weight_decay=0.05)
loader = torch.utils.data.DataLoader(X_train, batch_size=256, shuffle=True)   # images only: no labels

EPOCHS = 8
for epoch in range(EPOCHS):
    total = 0.0
    for images in loader:
        loss, _, _ = mae(images.to(DEVICE))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * len(images)
    print(f"epoch {epoch + 1}: reconstruction loss {total / len(X_train):.4f}")

Reconstructions on test tiles (never seen in training). The model fills in hidden patches from the visible ones:

In [ ]:
mae.eval()
with torch.no_grad():
    images = X_test[:6].to(DEVICE)
    _, predicted, hidden = mae(images)
    shown = patchify(images).clone()
    for i in range(len(images)):
        shown[i, hidden[i]] = predicted[i, hidden[i]]                       # paste predictions into hidden spots
    masked = patchify(images).clone()
    for i in range(len(images)):
        masked[i, hidden[i]] = 0.8

fig, axes = plt.subplots(3, 6, figsize=(9, 4.8))
for j in range(6):
    for r, (img, title) in enumerate([(images[j], "original"), (unpatchify(masked)[j], "input (75% hidden)"),
                                      (unpatchify(shown)[j], "reconstruction")]):
        axes[r, j].imshow(img.cpu().clamp(0, 1).permute(1, 2, 0)); axes[r, j].axis("off")
        if j == 0: axes[r, j].set_title(title, fontsize=9, loc="left")
plt.tight_layout(); plt.show()

## 4. Contrastive learning  *(slides: "Two Views" to "The InfoNCE Loss")*

### 4.1 The InfoNCE loss with the slide numbers
Similarities of A1 to A2 (positive), B, C, D: 0.9, 0.2, −0.1, 0.3. Divide by the temperature τ = 0.5, apply softmax, and take
minus the log of the positive's probability.

In [ ]:
similarities = torch.tensor([0.9, 0.2, -0.1, 0.3])     # A2 (positive), B, C, D
tau = 0.5
probabilities = torch.softmax(similarities / tau, dim=0)
print("probabilities:", probabilities.numpy().round(2))
print("loss for A1 = -log p(A2) =", round(-math.log(probabilities[0].item()), 2))

### 4.2 Two views of the same tile
Augmentations should change the **appearance**, not the **tissue type**. For pathology tiles, rotations and flips are safe
(tissue has no up or down); we keep color changes mild, because stain color carries information.

In [ ]:
def augment(images):
    # Random 90-degree rotation, random flips, mild brightness/contrast jitter, and a little noise. Same shape out.
    out = torch.rot90(images, k=int(torch.randint(0, 4, (1,))), dims=(2, 3))
    if torch.rand(1) < 0.5: out = out.flip(3)
    B = out.shape[0]
    brightness = 1 + 0.1 * (torch.rand(B, 1, 1, 1, device=out.device) - 0.5)
    contrast = 1 + 0.2 * (torch.rand(B, 1, 1, 1, device=out.device) - 0.5)
    mean = out.mean(dim=(1, 2, 3), keepdim=True)
    out = ((out - mean) * contrast + mean) * brightness
    return (out + 0.02 * torch.randn_like(out)).clamp(0, 1)

fig, axes = plt.subplots(1, 3, figsize=(5.5, 2))
for ax, (img, t) in zip(axes, [(X_train[0], "tile"), (augment(X_train[:1])[0], "view 1"), (augment(X_train[:1])[0], "view 2")]):
    ax.imshow(img.permute(1, 2, 0)); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()

### 4.3 A tiny SimCLR
A small CNN encoder maps each view to a vector. For a batch of N tiles we have 2N views; for each view, its partner is the
positive and the other 2N − 2 views are negatives. The loss is InfoNCE over the 2N × 2N similarity matrix.

In [ ]:
class SmallCNN(nn.Module):
    # Encoder: 28x28x3 tile -> 128-dim embedding.
    def __init__(self, dim=128):
        super().__init__()
        block = lambda cin, cout: nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.out_dim = 128
    def forward(self, x):
        return self.features(x)

def info_nce(z1, z2, tau=0.5):
    # InfoNCE (NT-Xent) loss for two batches of embeddings where row i of z1 and row i of z2 are the positive pair.
    z = F.normalize(torch.cat([z1, z2]), dim=1)          # 2N unit vectors
    sim = z @ z.T / tau                                   # 2N x 2N cosine similarities / temperature
    sim.fill_diagonal_(float("-inf"))                     # a view is not its own positive
    N = z1.shape[0]
    positives = torch.cat([torch.arange(N, 2 * N), torch.arange(0, N)]).to(z.device)   # partner index for each row
    return F.cross_entropy(sim, positives)                # softmax over each row, -log p(partner)

encoder_cl = SmallCNN().to(DEVICE)
projector = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 64)).to(DEVICE)  # discarded after training, like the MAE decoder
optimizer = torch.optim.AdamW(list(encoder_cl.parameters()) + list(projector.parameters()), lr=1e-3)

EPOCHS = 6
for epoch in range(EPOCHS):
    total = 0.0
    for images in loader:                                  # still no labels
        images = images.to(DEVICE)
        loss = info_nce(projector(encoder_cl(augment(images))), projector(encoder_cl(augment(images))))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * len(images)
    print(f"epoch {epoch + 1}: InfoNCE loss {total / len(X_train):.3f}")

## 5. Using the encoder: linear probes  *(slides: "Three Ways to Use a Pretrained Encoder" and "Label Efficiency")*

Freeze each encoder, turn every tile into an embedding, and train only a **linear classifier** (logistic regression) on
1%, 10%, and 100% of the training labels. We compare:
- a **random** encoder (same CNN, never trained): the baseline,
- the **MAE** encoder (mean of its patch vectors),
- the **contrastive** encoder.

In [ ]:
@torch.no_grad()
def embed(encode_fn, images, batch=512):
    # Apply a frozen encoder to all images and return a numpy array of embeddings.
    return np.concatenate([encode_fn(images[i:i + batch].to(DEVICE)).cpu().numpy() for i in range(0, len(images), batch)])

mae.eval(); encoder_cl.eval()
random_cnn = SmallCNN().to(DEVICE).eval()
encoders = {
    "random encoder": lambda x: random_cnn(x),
    "MAE encoder": lambda x: mae.encode(x).mean(dim=1),       # average the patch vectors
    "contrastive encoder": lambda x: encoder_cl(x),
}

def linear_probe(train_features, test_features, fraction, seed=0):
    # Train logistic regression on a random fraction of the labeled training set; return test accuracy.
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(train_features), max(int(fraction * len(train_features)), 2 * N_CLASSES), replace=False)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(train_features[idx], y_train.numpy()[idx])
    return (clf.predict(test_features) == y_test.numpy()).mean()

fractions = [0.01, 0.1, 1.0]
results = {}
for name, fn in encoders.items():
    ftr, fte = embed(fn, X_train), embed(fn, X_test)
    mu, sd = ftr.mean(0), ftr.std(0) + 1e-6                     # standardize features
    results[name] = [linear_probe((ftr - mu) / sd, (fte - mu) / sd, f) for f in fractions]
    print(f"{name:>20s}: " + "  ".join(f"{int(f * 100):>3d}% labels -> {a:.3f}" for f, a in zip(fractions, results[name])))

In [ ]:
plt.figure(figsize=(6, 3.5))
width = 0.25
for k, (name, accs) in enumerate(results.items()):
    plt.bar(np.arange(3) + (k - 1) * width, accs, width, label=name)
plt.xticks(range(3), ["1% of labels", "10% of labels", "100% of labels"])
plt.ylabel("test accuracy (linear probe)"); plt.ylim(0, 1); plt.legend(fontsize=8)
plt.title("Label efficiency on PathMNIST"); plt.tight_layout(); plt.show()

**What to look for:** both self-supervised encoders should beat the random encoder, and the gap is usually largest with 1% of
the labels. With tiny models and a few minutes of training the numbers are modest; the lecture's foundation models use the same
recipe at a vastly larger scale.

## 6. Optional: features from a pretrained imaging foundation model  *(slides: "Imaging Foundation Models")*

Pathology foundation models such as UNI are gated on Hugging Face (you must request access and agree to the terms), so here
we use **DINOv2-small**, the general-purpose model trained with the same self-distillation recipe that UNI uses. We resize the
tiles to 224 × 224, take the `[CLS]` vector, and run the same linear probe. Set `RUN_DINOV2 = False` to skip.

In [ ]:
RUN_DINOV2 = True
if RUN_DINOV2:
    try:
        from transformers import AutoModel
        dino = AutoModel.from_pretrained("facebook/dinov2-small").to(DEVICE).eval()
        mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)   # ImageNet normalization used by DINOv2
        std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
        def dino_features(x):
            x = F.interpolate(x, size=224, mode="bilinear", align_corners=False)
            return dino(pixel_values=(x - mean) / std).last_hidden_state[:, 0]          # [CLS] vector
        n = 2000                                                                       # a subset keeps CPU time reasonable
        ftr, fte = embed(dino_features, X_train[:n], batch=64), embed(dino_features, X_test[:1000], batch=64)
        y_tr, y_te = y_train.numpy()[:n], y_test.numpy()[:1000]
        for f in [0.05, 1.0]:
            idx = np.random.default_rng(0).choice(n, int(f * n), replace=False)
            acc = (LogisticRegression(max_iter=3000).fit(ftr[idx], y_tr[idx]).predict(fte) == y_te).mean()
            print(f"DINOv2-small linear probe, {int(f * n)} labels: test accuracy {acc:.3f}")
    except Exception as error:
        print("Skipped DINOv2 section:", type(error).__name__, error)

## Recap

| Idea | Code in this notebook |
|---|---|
| Patches and masking | `patchify`, `random_mask` (75% hidden) |
| Masked reconstruction | `TinyMAE`: encoder on visible patches, mask tokens, loss on hidden patches |
| Contrastive learning | `augment`, `info_nce` over the 2N × 2N similarity matrix |
| Linear probe | frozen `embed` + `LogisticRegression` on 1%, 10%, 100% of labels |
| Foundation model features | DINOv2-small `[CLS]` vectors |

> **Try it (optional):** change `mask_ratio` to 0.25 or 0.9 and compare linear-probe accuracy. Remove the rotation from
> `augment`. Add strong color jitter: does accuracy drop, as the lecture warns?